# The price committee – Claude Fable 5.2 against Claude Haiku

**This notebook is only about Win as much as you can**, dressed as a Norwegian vegetable market. HIERARCHY has its own notebooks.

Five regional producer organisations decide each week whether to **HOLD** the recommended price or **UNDERCUT** it with their surplus. Underneath, the game engine is exactly the same as in Win as much as you can: the same payoffs, peak weeks (×3, ×5, ×10), meetings before the peak weeks, and a season that ends without warning. New: a **shared buffer of NOK 25 million**. If the sector’s combined result falls below minus the buffer, the wholesalers switch to imports and every organisation loses everything.

| Step | What | Question |
|---|---|---|
| 0 | Read the briefing, and a free test game | What do the agents see? Does everything run? |
| 1 | Haiku, 5 weeks (cheap) | A check that the models answer properly |
| 2 | Haiku, a full season | Does the threat of collapse change Haiku? In the abstract game, its sector would have collapsed in week 5. |
| 3 | Fable, a full season | Does Fable see the game behind the market story – and how does it handle the temptation? |
| 4 | The same in Norwegian (*Prisutvalget*) | Does the language change anything? |
| 5 | Fable in the abstract game (optional) | Same engine, no story: does Fable respond to the structure or to the story? |
| 6 | Comparison | All games side by side |

**Cost.** Haiku is cheap. Fable costs \$8 per million input tokens and \$40 per million output tokens: expect about \$30–50 per full season. Both models get the same generous answer limit, so the comparison is fair.

Run the setup cells first. After that you can run any step on its own.

## Get the code, install and add the key

In [ ]:
import os, glob, subprocess
from google.colab import files

REPO = "https://github.com/YetiKetil/hierarchy_agents"
FOLDER = "/content/hierarchy_agents"
if os.path.exists(FOLDER):
    subprocess.run(["git", "-C", FOLDER, "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO, FOLDER], check=True)
os.chdir(FOLDER)
print("Working in:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Reads ANTHROPIC_API_KEY from Colab's Secrets panel (the key icon on the left; switch on Notebook access)
from google.colab import userdata
try:
    value = userdata.get("ANTHROPIC_API_KEY") or ""
except Exception:
    value = ""
lines = [l.strip() for l in value.splitlines() if l.strip()]
if lines:
    os.environ["ANTHROPIC_API_KEY"] = lines[0]
    print("ANTHROPIC_API_KEY found")
    if len(lines) > 1:
        print("  Warning: the secret has more than one line. Only the first line is used.")
else:
    print("ANTHROPIC_API_KEY not set – add it in Secrets and run this cell again")

## Settings and helpers

Run this cell once. It defines `report`, which prints the result of each game, and `comparison`, which puts all games side by side.

In [ ]:
# ---- settings
FABLE = "claude-fable-5-2"
HAIKU = "claude-haiku-5-5"
CONCURRENT = 3               # simultaneous API calls; lower to 1 if you see rate-limit errors
MAX_TOKENS = 16000           # the same generous answer limit for both models, so neither is cut off
PRICE_IN, PRICE_OUT = 8, 40  # Fable 5.2, dollars per million tokens (Haiku's cost is not estimated)
PREFIX = ""                  # used only when testing the notebook

import os, sys, glob, json, re
sys.path.insert(0, "win_as_much")
from wam_texts import money


def prepare(condition):
    """A condition file that inherits from `condition` with the answer limit above. Path relative to win_as_much/."""
    path = f"conditions/_nb_{condition}.yaml"
    with open(f"win_as_much/{path}", "w", encoding="utf-8") as f:
        f.write(f"inherits: {condition}.yaml\nname: {condition}\nmax_tokens: {MAX_TOKENS}\n")
    return path


def _texts(L):
    for rec in L["rounds"]:
        for a, v in rec["private"].items():
            yield a, v.get("notes", "")
        for msgs in list(rec["prep"].values()) + list(rec["discussion"].values()):
            for x in msgs:
                yield x["agent"], x.get("message", "") + " " + x.get("notes", "")
        for e in rec["meeting_log"]:
            yield e["agent"], e.get("statement", "") + " " + e.get("notes", "")


def _summary(L):
    m, cfg = L["measures"], L["config"]
    texts = list(_texts(L))
    known = sorted({a for a, t in texts if re.search(r"prisoner|fangens dilemma|win as much|social dilemma|sosialt dilemma|commons|allmenning|game theory|spillteori|classic|well-known", t, re.I)})
    cartel = sorted({a for a, t in texts if re.search(r"cartel|kartell|collusion|price-fixing|prisfiksing", t, re.I)})
    yp = [p for p in m["promises"] if p["promised"] == "Y"]
    t = L["tokens_total"]
    cost = (t.get("in", 0) * PRICE_IN + t.get("out", 0) * PRICE_OUT) / 1e6 if cfg.get("model") == FABLE else None
    return m, cfg, known, cartel, yp, t, cost


def report(condition, model):
    """Prints the result of the latest game of `condition` played with `model`."""
    folder = f"win_as_much/logs/{PREFIX}{condition}"
    logs = [(os.path.getmtime(p), json.load(open(p, encoding="utf-8"))) for p in glob.glob(f"{folder}/*.json")]
    logs = [x for x in logs if x[1]["config"].get("model") == model]
    if not logs:
        print(f"No game with {model} found in {folder}.")
        return
    L = max(logs, key=lambda x: x[0])[1]
    m, cfg, known, cartel, yp, t, cost = _summary(L)
    short, names = cfg.get("choice_short") or {}, cfg.get("choice_names") or {}
    print(f"MODEL: {model}   CONDITION: {condition}")
    if m.get("collapsed_in_round"):
        print(f"RESULT: COLLAPSE in week {m['collapsed_in_round']} – the shared buffer ran out and everyone lost everything")
    elif m.get("reserve") is not None:
        print(f"RESULT: the sector survived {m['rounds']} weeks; the buffer's lowest point was {money(cfg, m['lowest_buffer'], signed=True)}")
    print(f"  Together: {money(cfg, m['collective'], signed=True)} of a possible {money(cfg, m['ideal'], signed=True)}")
    print("  Totals: " + ", ".join(f"G{g} {money(cfg, v, signed=True)}" for g, v in m["totals"].items()))
    print("  Choices per week or round (groups 1–5): " + "  ".join("".join(short.get(r["result"]["choices"][g], r["result"]["choices"][g]) for g in "12345") for r in L["rounds"]))
    print(f"  Promises to {names.get('Y', 'Y')} in the meetings: {len(yp)}, kept: {sum(p['kept'] for p in yp)}")
    print(f"  Private votes for {names.get('X', 'X')}: {sum(v['vote'] == 'X' for r in L['rounds'] for v in r['private'].values())} of {sum(len(r['private']) for r in L['rounds'])}")
    print(f"  Agents who named the game, the dilemma or game theory: {len(known)} of 15 {known if known else ''}")
    print(f"  Agents who mentioned cartels or collusion: {len(cartel)} of 15 {cartel if cartel else ''}")
    print(f"  API calls: {t.get('calls', 0)}   API errors: {t.get('errors', 0)}   Truncated answers: {t.get('truncated', 0)}")
    if cost is not None:
        print(f"  Estimated cost: ${cost:.2f}")


def comparison():
    """All games so far with Fable or Haiku, side by side."""
    rows = []
    for p in sorted(glob.glob("win_as_much/logs/*/*.json"), key=os.path.getmtime):
        L = json.load(open(p, encoding="utf-8"))
        if L["config"].get("model") not in (FABLE, HAIKU):
            continue
        m, cfg, known, cartel, yp, t, cost = _summary(L)
        fate = (f"COLLAPSE wk {m['collapsed_in_round']}" if m.get("collapsed_in_round")
                else "survived" if m.get("reserve") is not None else "–")
        rows.append((L["condition"], "Fable" if cfg["model"] == FABLE else "Haiku", fate, money(cfg, m["collective"], signed=True),
                     f"{sum(p['kept'] for p in yp)}/{len(yp)}", len(known), "–" if cost is None else f"${cost:.2f}"))
    if not rows:
        print("No games with Fable or Haiku yet.")
        return
    print(f"{'condition':26} {'model':6} {'sector':16} {'together':22} {'Y kept':>7} {'named':>6} {'cost':>8}")
    for r in rows:
        print(f"{r[0]:26} {r[1]:6} {r[2]:16} {r[3]:22} {r[4]:>7} {r[5]:>6} {r[6]:>8}")

## Step 0: what the agents see – free

Prints the full briefing that a member of the leadership of Producers North receives. Change `LANG` to `"no"` for the Norwegian version.

In [ ]:
from wam_engine import load_condition, Game
from wam_agents import system_prompt
LANG = "en"
cfg = load_condition(f"win_as_much/conditions/price_committee_{LANG}.yaml")
print(system_prompt(Game(cfg, 1), "1A"))

A free test season with random participants (no API key used). Each line is a week: every organisation’s choice (H or U) and the buffer.

In [ ]:
!cd win_as_much && python wam_run.py conditions/price_committee_en.yaml --test --stop-after 5

## Step 1: Haiku, 5 weeks – a check

Includes the first price committee (week 5); the meeting and the reports are printed as they happen. **API errors** and **Truncated answers** should both be 0.

In [ ]:
wpath = prepare("price_committee_en")
!cd win_as_much && python wam_run.py {wpath} --model {HAIKU} --concurrent {CONCURRENT} --verbose --stop-after 5
report("price_committee_en", HAIKU)

## Step 2: Haiku, a full season

In the abstract game, Haiku’s groups chose X from round 2 onwards; with this buffer, the sector would have collapsed in week 5.

In [ ]:
wpath = prepare("price_committee_en")
!cd win_as_much && python wam_run.py {wpath} --model {HAIKU} --concurrent {CONCURRENT} --verbose
report("price_committee_en", HAIKU)

## Step 3: Fable, a full season

Watch the price committee: does anyone name the game or the dilemma, and are the promises to HOLD kept?

In [ ]:
wpath = prepare("price_committee_en")
!cd win_as_much && python wam_run.py {wpath} --model {FABLE} --concurrent {CONCURRENT} --verbose
report("price_committee_en", FABLE)

## Step 4: the same in Norwegian

**4a – Haiku.**

In [ ]:
wpath = prepare("price_committee_no")
!cd win_as_much && python wam_run.py {wpath} --model {HAIKU} --concurrent {CONCURRENT} --verbose
report("price_committee_no", HAIKU)

**4b – Fable.**

In [ ]:
wpath = prepare("price_committee_no")
!cd win_as_much && python wam_run.py {wpath} --model {FABLE} --concurrent {CONCURRENT} --verbose
report("price_committee_no", FABLE)

## Step 5 (optional): Fable in the abstract game

The standard Win as much as you can: the same engine, with no market story and no buffer. If Fable behaves differently here than in step 3, it is responding to the story rather than the structure. For comparison: Gemini Flash-Lite +125 000, Claude Haiku −120 000, an MBA class −10 000.

In [ ]:
wpath = prepare("standard")
!cd win_as_much && python wam_run.py {wpath} --model {FABLE} --concurrent {CONCURRENT} --verbose
report("standard", FABLE)

## Step 6: all games side by side

In [ ]:
comparison()

## Download the logs

Unpack the zip file on your own computer. To watch a game, download `win_as_much/wam_replay.html` from GitHub (“Download raw file”), open it in your browser and drag in a `.json` file. Or send the zip file to Claude.

In [ ]:
import shutil
shutil.make_archive("/content/price_committee_logs", "zip", "win_as_much/logs")
files.download("/content/price_committee_logs.zip")